# Ridge baseline

Minimal starter notebook that trains a Ridge model and writes `predicted.json` plus `predicted.zip` for submission.


In [9]:
import json
import zipfile
from pathlib import Path

import numpy as np
from sklearn.linear_model import Ridge
from sklearn.preprocessing import MinMaxScaler

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "baseline" else Path.cwd()
BASELINE_DIR = PROJECT_ROOT / "baseline"
DATASET_DIR = PROJECT_ROOT / "dataset"

FEATURE_KEYS = ["built_area", "num_lots", "num_commercial"]
TARGET_KEY = "property_value"

TRAIN_PATH = DATASET_DIR / "train.json"
TEST_PATH = DATASET_DIR / "test.json"
PREDICTION_JSON_PATH = BASELINE_DIR / "predicted.json"
PREDICTION_ZIP_PATH = BASELINE_DIR / "predicted.zip"

In [7]:
def load_json(path):
    """Load a JSON file and return the parsed Python object."""
    with open(path, "r", encoding="utf-8") as handle:
        return json.load(handle)


def make_feature_matrix(records):
    return np.array(
        [[float(record.get(key, 0.0) or 0.0) for key in FEATURE_KEYS] for record in records],
        dtype=np.float64,
    )


def fit_ridge_baseline(train_records, target_key=TARGET_KEY, alpha=10.0):
    """Fit the Ridge baseline on the starter features."""
    x_train = make_feature_matrix(train_records)
    y_train = np.array([record[target_key] for record in train_records], dtype=np.float64)

    scaler = MinMaxScaler()
    x_train = scaler.fit_transform(x_train)

    model = Ridge(alpha=alpha)
    model.fit(x_train, y_train)
    return model, scaler


def predict_records(model, scaler, records):
    """Predict property values for prepared records."""
    x_test = scaler.transform(make_feature_matrix(records))
    return [float(value) for value in model.predict(x_test)]

In [8]:
train_records = load_json(TRAIN_PATH)
test_records = load_json(TEST_PATH)

model, scaler = fit_ridge_baseline(train_records, target_key=TARGET_KEY, alpha=10.0)
predictions = [
    {TARGET_KEY: value}
    for value in predict_records(model, scaler, test_records)
]

PREDICTION_JSON_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(PREDICTION_JSON_PATH, "w", encoding="utf-8") as handle:
    json.dump(predictions, handle, indent=2)

with zipfile.ZipFile(PREDICTION_ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(PREDICTION_JSON_PATH, arcname="predicted.json")

print(f"Wrote {len(predictions)} predictions using: {', '.join(FEATURE_KEYS)}")

FileNotFoundError: [Errno 2] No such file or directory: 'dataset\\train.json'